In [1]:
%run 05_MNESIS_parameters.ipynb

datetag = '2026-04-21'
SNNtorch version 0.9.4
Spikes in one target 1024.0,  in a SM window 42.0
for a value opt.lif_beta=0.7, the time constant is 2.8 steps


In [2]:
opt = Params()
opt

Params(datetag='2026-04-21', N_neuron=1024, num_delay=41, N_SM=16, N_time=1000, N_pretime=50, p_A=0.001, seed=2018, lif_beta=0.7, learn_beta=False, learn_threshold=False, do_deconv=False, num_epochs=16, num_warmup_epochs=16, base_lr=0.001, final_lr=0.0001, delta1=0.01, delta2=5e-05, weight_decay=1e-09, init_gain=1.5, dropout=0.37, alpha_surrogate=15.0, surrogate_name='FastSigmoid', loss_name='SpikeF1scoreLoss', reset_mechanism='subtract', optimizer='sgd', verbose=False, fig_width=15, fig_height=9, phi=1.61803, N_time_show=400, N_neuron_show=256, N_scan=5)

In [3]:
print(f'Spikes in one target {opt.p_A * opt.N_neuron * opt.N_time:.1f},  in a SM window {opt.p_A * opt.N_neuron * opt.num_delay:.1f}')

Spikes in one target 1024.0,  in a SM window 42.0


In [4]:
print(f'for a value {opt.lif_beta=:.1f}, the time constant is {- 1 / np.log(opt.lif_beta):.1f} steps')

for a value opt.lif_beta=0.7, the time constant is 2.8 steps


## compute scores

In [5]:
def get_scores(pred, target, epsilon=1e-12):
    TP = (pred * target).sum() # Count elements where both are 1
    FP = (pred * (1 - target)).sum() # Count elements where pred is 1, target is 0
    FN = ((1-pred) * target).sum()  # Count elements where pred is 0, target is 1
    precision = TP / (TP + FP + epsilon) # when minimizing false positives is critical 
    recall = TP / (TP + FN + epsilon) # when minimizing false negatives is critical 
    f1_score = 2 * (precision * recall) / (precision + recall + epsilon)
    return precision, recall, f1_score

def get_f1score(pred, target, epsilon=1e-12):
    _, _, f1_score = get_scores(pred, target, epsilon=epsilon)
    return f1_score

class SpikeF1scoreLoss(nn.Module):
    # https://en.wikipedia.org/wiki/Dice-Sørensen_coefficient
    def __init__(self, epsilon=1e-12):
        super().__init__()
        self.epsilon = epsilon

    def forward(self, pred, target):
        f1_score =  get_f1score(pred, target, self.epsilon)
        return 1-f1_score
loss_fn = SpikeF1scoreLoss()

# print(f'Initial loss : {loss_fn(spikes.to(device), a_sample.to(device)).item():.3f}')

# SnnTorch model

In [6]:
def get_cosine_schedule_with_warmup(optimizer, num_warmup_epochs, num_epochs, rel_final_lr):
    def lr_lambda(current_epoch):
        if current_epoch < num_warmup_epochs:
            # Constant warmup of 1
            return 1
        else:
            # Cosine decay from base_lr to final_lr
            progress = (current_epoch - num_warmup_epochs) / max(1, num_epochs - num_warmup_epochs)
            cosine_decay = 0.5 * (1 + np.cos(np.pi * progress)) # from 1 to zero
            return (cosine_decay + rel_final_lr) / (1 + rel_final_lr) # between 1 and down to rel_final_lr

    scheduler = LambdaLR(optimizer, lr_lambda, last_epoch=-1)
    return scheduler

In [7]:
class HD_SNN(nn.Module):
    def __init__(self, opt):
        super().__init__()
        self.opt = opt
        # self.env = ABCD(opt)
        # self.logit_A = self.env.model_a_logit().to(self.opt.device)
        target_generator = torch.Generator()
        target_generator.manual_seed(opt.seed)
        p_bias = opt.p_A * torch.ones((opt.N_SM, opt.N_neuron, opt.N_time))
        self.target = torch.bernoulli(p_bias, generator=target_generator)
        self.target = self.target.float()
        self.target = self.target.to(opt.device)

        dropout = nn.Dropout(opt.dropout)

        lin = nn.Linear(opt.num_delay*opt.N_neuron, opt.N_neuron, bias=False)
        if self.opt.surrogate_name == 'FastSigmoid':
            spike_grad = surrogate.fast_sigmoid(slope=opt.alpha_surrogate)
        elif self.opt.surrogate_name == 'LeakySpikeOperator':
            spike_grad = surrogate.LSO(slope=opt.alpha_surrogate)
        elif self.opt.surrogate_name == 'ATan':
            spike_grad = surrogate.atan(alpha=opt.alpha_surrogate)
        elif self.opt.surrogate_name == 'SpikeRateEscape':
            spike_grad = surrogate.spike_rate_escape(slope=opt.alpha_surrogate)
        elif self.opt.surrogate_name == 'Sigmoid':
            spike_grad = surrogate.sigmoid(slope=opt.alpha_surrogate)

        lif = snn.Leaky(beta=opt.lif_beta, 
                        learn_beta=opt.learn_beta, learn_threshold=opt.learn_threshold, output=False, # init_hidden=True, 
                        reset_mechanism=opt.reset_mechanism, spike_grad=spike_grad)

        net = nn.Sequential(OrderedDict([('dropout', dropout), ('lin', lin), ('lif', lif) ]))
        net = net.to(opt.device)
        self.net = net
          
    def forward_pass(self, input_spikes):
        input_spikes = input_spikes.to(self.opt.device).detach()
        assert not input_spikes.requires_grad, "input_spikes still requires grad!"

        with torch.no_grad():
            snn_utils.reset(self.net)

        device, dtype = self.opt.device, torch.float32

        N_SM = input_spikes.shape[0]
        N_time = input_spikes.shape[-1] # self.opt.N_time+2*self.opt.N_pretime
        current = torch.zeros(N_SM, self.opt.N_neuron, N_time, device=device, dtype=dtype)
        spikes  = torch.zeros(N_SM, self.opt.N_neuron, N_time, device=device, dtype=dtype)
        mem_rec = torch.zeros(N_SM, self.opt.N_neuron, N_time, device=device, dtype=dtype)
        # Initalize membrane potential
        mem = self.net.lif.init_leaky() # pyright: ignore[reportCallIssue, reportAttributeAccessIssue]

        for t in range(self.opt.num_delay, N_time):
            spike_window_A = spikes[:, :, (t - self.opt.num_delay):t]
            spike_window_B = input_spikes[:, :, (t - self.opt.num_delay):t]
            # print(t, spike_window_A.shape, spike_window_B.shape)
            # spike_window = torch.max(spike_window_A, spike_window_B)
            spike_window = (spike_window_A + spike_window_B).clamp(0, 1)
            # window   [N_SM, N_neuron, num_delay]  ->  [N_SM, N_neuron * num_delay]
            raveled_spks = spike_window.reshape(N_SM, self.opt.N_neuron * self.opt.num_delay)
            cur = self.net.lin(raveled_spks)                     # pyright: ignore[reportCallIssue] # shape [1, N_neuron]
            # HACK 
            cur = self.net.dropout(cur) # pyright: ignore[reportCallIssue]

            spk, mem = self.net.lif(cur, mem)          # pyright: ignore[reportCallIssue] # spk, mem shape [1, N_neuron]
            
            current[:, :, t] = cur
            spikes[:, :, t] = spk
            mem_rec[:, :, t] = mem

        return current, mem_rec, spikes

    def update_weight(self):
        with torch.no_grad():
            # Collect all context-target pairs - see `2026-04-08_MNESIS_mulltiPC` for details
            windows = self.target[:, :, :-1].unfold(dimension=2, size=self.opt.num_delay, step=1)
            windows = windows.permute(0, 2, 1, 3).contiguous()
            batch =  self.opt.N_SM * (self.opt.N_time - self.opt.num_delay)
            contexts = windows.reshape(batch,  self.opt.N_neuron *  self.opt.num_delay)
            if self.opt.do_deconv:
                deconvolved_target = self.target - self.opt.lif_beta * torch.roll(self.target, 1, dims=-1)
                targets = deconvolved_target[:, :,  self.opt.num_delay:]             # shape (N_time-num_delay, N_SM, N_neuron)
            else:
                targets = self.target[:, :,  self.opt.num_delay:]             # shape (N_time-num_delay, N_SM, N_neuron)
            targets = targets.permute(0, 2, 1)       # (N_SM, N_time-num_delay, N_neuron)
            targets = targets.reshape(batch, self.opt.N_neuron)

            # TODO: use and compare the other method with simply using the correlation
            # Compute pseudo-inverse: W = (X^T X)^(-1) X^T Y
            X_pinv = torch.linalg.pinv(contexts)  # [N_neuron * num_delay, total_samples]
            W_flat = torch.matmul(X_pinv, targets)  # [N_neuron * num_delay, N_neuron]
            
            # Transpose to get [N_neuron, N_neuron * num_delay]
            W_flat = W_flat.transpose(0, 1)
            
            # Copy to network weights
            self.net.lin.weight.copy_(self.opt.init_gain * W_flat) # pyright: ignore[reportCallIssue, reportAttributeAccessIssue]


    def get_input_spikes(self, p_A, N_pretime, N_trigger_time, N_time):
        # clamp spikes to start the chain
        # 1/ empty spike list
        input_spikes = torch.zeros((self.opt.N_SM, self.opt.N_neuron, N_time+2*N_pretime))
        # 2/ spontaneous activity
        input_spikes[:, :, :N_pretime] = torch.bernoulli(p_A * torch.ones((self.opt.N_SM, self.opt.N_neuron, N_pretime)))
        # 3/ target activity           
        input_spikes[:, :, N_pretime:(N_pretime+N_trigger_time)] = self.target[:, :, :N_trigger_time]

        return input_spikes.to(self.opt.device).detach()

    def learn_model(self, verbose=True):
        
        if self.opt.loss_name == 'SpikeF1scoreLoss':
            loss_fn = SpikeF1scoreLoss()
        elif self.opt.loss_name == 'MSELoss':
            loss_fn = nn.MSELoss()

        self.net = self.net.to(device)
        
        
        optimizer_dict = dict(lr=self.opt.base_lr, weight_decay=self.opt.weight_decay)
        if self.opt.optimizer=='adam': 
            optimizer = torch.optim.Adam(self.net.parameters(), betas=(1-self.opt.delta1, 1-self.opt.delta2), **optimizer_dict)
        elif self.opt.optimizer=='adamw': 
            optimizer = torch.optim.AdamW(self.net.parameters(), betas=(1-self.opt.delta1, 1-self.opt.delta2), **optimizer_dict)
        elif self.opt.optimizer=='sparseadam': 
            optimizer = torch.optim.AdamW(self.net.parameters(), betas=(1-self.opt.delta1, 1-self.opt.delta2), **optimizer_dict)
        elif self.opt.optimizer=='sgd': 
            optimizer = torch.optim.SGD(self.net.parameters(),  momentum=1-self.opt.delta1, dampening=1-self.opt.delta2, **optimizer_dict)
        elif self.opt.optimizer=='rmsprop': 
            optimizer = torch.optim.RMSprop(self.net.parameters(), momentum=1-self.opt.delta1, alpha=1-self.opt.delta2, **optimizer_dict)
        elif self.opt.optimizer=='adadelta': 
            optimizer = torch.optim.Adadelta(self.net.parameters(), rho=1-self.opt.delta1, **optimizer_dict)
        else:
            raise(ValueError(f'Unknown optimizer {self.opt.optimizer}'))

        scheduler = get_cosine_schedule_with_warmup(optimizer,
                                                    self.opt.num_warmup_epochs, self.opt.num_epochs, self.opt.final_lr/self.opt.base_lr)

        loss_val, precision, recall, f1_score = [], [], [], []
        log_interval = max(self.opt.num_epochs // 64, 1)

        for i_step in range(self.opt.num_epochs):
            self.net.train()

            # start optimization
            optimizer.zero_grad()
            input_spikes = self.get_input_spikes(p_A=self.opt.p_A, N_pretime=self.opt.N_pretime, N_trigger_time=self.opt.num_delay, N_time=self.opt.N_time).detach()
            _, _, spikes = self.forward_pass(input_spikes)
            loss_train = loss_fn(spikes[:, :, (self.opt.N_pretime+self.opt.num_delay):(-self.opt.N_pretime)], 
                                 self.target[:, :, self.opt.num_delay:])
            loss_train.backward()
            optimizer.step()
            scheduler.step()

            with torch.no_grad():
                self.net.eval()
                # draw causes (SMs) - hidden to the observer
                input_spikes = self.get_input_spikes(p_A=self.opt.p_A, N_pretime=self.opt.N_pretime, N_trigger_time=self.opt.num_delay, N_time=self.opt.N_time)
                _, _, spikes = self.forward_pass(input_spikes)
                spikes_ = spikes[:, :, (self.opt.N_pretime+self.opt.num_delay):(-self.opt.N_pretime)]
                target_ = self.target[:, :, self.opt.num_delay:]
                loss_val_ = loss_fn(spikes_, target_)
                loss_val.append(loss_val_.item())
                precision_, recall_, f1_score_ = get_scores(spikes_, target_)
                precision.append(precision_.cpu()) 
                recall.append(recall_.cpu())
                f1_score.append(f1_score_.cpu())

            if verbose and ((i_step + 1) % log_interval == 0):
                print(f'Train Epoch [{i_step+1:06d}/{self.opt.num_epochs:06d}]\t| Loss = {np.mean(loss_val):.3e}\t| precision = {np.mean(precision):.3f}\t| recall = {np.mean(recall):.3f}\t| f1_score = {np.mean(f1_score):.3f}\t| ')
                loss_val, precision, recall, f1_score = [], [], [], []
          